# 📊 Customer Shopping Behavior Analysis & Predictive Modeling
**Author:** Aditya Tomar  
**Tech Stack:** Python (Pandas, NumPy, Matplotlib, Seaborn, Scikit-Learn, SQLAlchemy)  

--- 

## 📌 Overview
This notebook provides a complete data science workflow analyzing retail customer behavior across 3,900 purchase transactions. Key objectives:
1. **Data Cleaning & Preprocessing**: Handling missing review ratings and standardizing schema.
2. **Feature Engineering**: Binning age groups, mapping purchase frequencies, and RFM scoring.
3. **Exploratory Data Analysis (EDA)**: Uncovering revenue patterns, discount sensitivity, and demographic spending.
4. **RFM Customer Segmentation**: Segmenting customers into actionable behavioral tiers.
5. **Predictive Machine Learning**: Training a Random Forest classifier to predict **Subscription Status**.
6. **Database Migration**: Exporting processed datasets into PostgreSQL/MySQL.

## 1. Environment Setup & Data Loading

In [ ]:
# Import core data processing and visualization libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Machine Learning libraries
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve

import warnings
warnings.filterwarnings('ignore')

# Plotting configuration
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (10, 6)

# Load dataset
df = pd.read_csv('customer_shopping_behavior.csv')
df.head()

## 2. Exploratory Data Inspection & Data Cleaning

In [ ]:
# Inspect dataset shape and column data types
print(f"Dataset Dimensions: {df.shape[0]} rows, {df.shape[1]} columns")
df.info()

In [ ]:
# Statistical summary of numeric and categorical attributes
df.describe(include='all')

In [ ]:
# Check missing values
print("Missing values per column:")
print(df.isnull().sum())

# Impute missing 'Review Rating' values with median rating of the corresponding product Category
df['Review Rating'] = df.groupby('Category')['Review Rating'].transform(lambda x: x.fillna(x.median()))
print("\nMissing values after imputation:", df['Review Rating'].isnull().sum())

In [ ]:
# Standardize column names to snake_case
df.columns = df.columns.str.lower().str.replace(' ', '_').str.replace('(', '').str.replace(')', '')
df = df.rename(columns={'purchase_amount_usd': 'purchase_amount'})
print("Cleaned Columns:", df.columns.tolist())

## 3. Feature Engineering

In [ ]:
# 1. Create Age Groups (Young Adult, Adult, Middle-aged, Senior)
age_labels = ['Young Adult', 'Adult', 'Middle-aged', 'Senior']
df['age_group'] = pd.qcut(df['age'], q=4, labels=age_labels)

# 2. Map purchase frequency strings to numerical days
frequency_mapping = {
    'Weekly': 7,
    'Bi-Weekly': 14,
    'Fortnightly': 14,
    'Monthly': 30,
    'Every 3 Months': 90,
    'Quarterly': 90,
    'Annually': 365
}
df['purchase_frequency_days'] = df['frequency_of_purchases'].map(frequency_mapping)

# 3. Drop redundant promo_code_used column (identical to discount_applied)
if 'promo_code_used' in df.columns:
    df = df.drop(columns=['promo_code_used'])

df[['age', 'age_group', 'frequency_of_purchases', 'purchase_frequency_days']].head()

## 4. Advanced Exploratory Data Analysis (EDA)

In [ ]:
# Revenue & Order Distribution across Product Categories
plt.figure(figsize=(12, 5))
sns.boxplot(data=df, x='category', y='purchase_amount', palette='Set2')
plt.title('Purchase Amount Distribution by Product Category', fontsize=14, fontweight='bold')
plt.xlabel('Category', fontsize=12)
plt.ylabel('Purchase Amount ($)', fontsize=12)
plt.show()

In [ ]:
# Correlation Heatmap among Numerical Features
numerical_cols = ['age', 'purchase_amount', 'review_rating', 'previous_purchases', 'purchase_frequency_days']
plt.figure(figsize=(8, 6))
sns.heatmap(df[numerical_cols].corr(), annot=True, cmap='Blues', fmt='.2f', linewidths=0.5)
plt.title('Numerical Feature Correlation Matrix', fontsize=14, fontweight='bold')
plt.show()

In [ ]:
# Subscription vs Non-Subscription Spend Breakdown
plt.figure(figsize=(10, 5))
sns.barplot(data=df, x='category', y='purchase_amount', hue='subscription_status', ci=None, palette='viridis')
plt.title('Average Spend per Category: Subscribers vs Non-Subscribers', fontsize=14, fontweight='bold')
plt.xlabel('Category', fontsize=12)
plt.ylabel('Avg Purchase Amount ($)', fontsize=12)
plt.show()

## 5. RFM (Recency, Frequency, Monetary) Customer Segmentation

In [ ]:
# Calculate RFM Scores
# Recency Proxy: purchase_frequency_days (Lower days = More recent/frequent)
# Frequency: previous_purchases
# Monetary: purchase_amount

df['f_score'] = pd.qcut(df['previous_purchases'], q=4, labels=[1, 2, 3, 4])
df['m_score'] = pd.qcut(df['purchase_amount'], q=4, labels=[1, 2, 3, 4])
df['rfm_total'] = df['f_score'].astype(int) + df['m_score'].astype(int)

def segment_customer(score):
    if score >= 7:
        return 'Champion / High-Value'
    elif score >= 5:
        return 'Loyal Customer'
    elif score >= 3:
        return 'Potential Loyalist'
    else:
        return 'At Risk'

df['customer_segment'] = df['rfm_total'].apply(segment_customer)

# Segment breakdown
segment_counts = df['customer_segment'].value_counts()
print("Customer Segment Distribution:")
print(segment_counts)

plt.figure(figsize=(9, 5))
sns.barplot(x=segment_counts.index, y=segment_counts.values, palette='crest')
plt.title('Customer Behavioral Segmentation Counts', fontsize=14, fontweight='bold')
plt.ylabel('Customer Count', fontsize=12)
plt.show()

## 6. Predictive Machine Learning: Subscription Status Prediction

In [ ]:
# Prepare features (X) and target (y)
ml_df = df.copy()

# Encode Target Variable: Subscription Status (Yes=1, No=0)
ml_df['target'] = (ml_df['subscription_status'] == 'Yes').astype(int)

# Select predictor features
feature_cols = ['age', 'purchase_amount', 'review_rating', 'previous_purchases', 
                'purchase_frequency_days', 'gender', 'category', 'shipping_type', 
                'discount_applied', 'payment_method']

X = ml_df[feature_cols]
y = ml_df['target']

# One-Hot Encode Categorical Features
X = pd.get_dummies(X, drop_first=True)

# Split into Train and Test sets (80% Train, 20% Test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Scale numerical features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Train Random Forest Classifier
rf_model = RandomForestClassifier(n_estimators=100, max_depth=8, random_state=42)
rf_model.fit(X_train, y_train)

# Predictions & Probabilities
y_pred = rf_model.predict(X_test)
y_probs = rf_model.predict_proba(X_test)[:, 1]

print("Model Training Complete.")

In [ ]:
# Model Evaluation: Classification Report & Confusion Matrix
print("=== Classification Report ===")
print(classification_report(y_test, y_pred))

cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['No Sub', 'Sub'], yticklabels=['No Sub', 'Sub'])
plt.title('Confusion Matrix: Subscription Prediction', fontsize=13, fontweight='bold')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.show()

In [ ]:
# Feature Importance Analysis
importances = pd.Series(rf_model.feature_importances_, index=X.columns).sort_values(ascending=False).head(10)

plt.figure(figsize=(10, 5))
sns.barplot(x=importances.values, y=importances.index, palette='mako')
plt.title('Top 10 Feature Importances Driving Subscription Likelihood', fontsize=14, fontweight='bold')
plt.xlabel('Importance Score')
plt.show()

## 7. Database Export Setup

In [ ]:
from sqlalchemy import create_engine
import os

# Parameterized connection string (replaces hardcoded credentials)
username = os.getenv("DB_USER", "postgres")
password = os.getenv("DB_PASS", "your_password")
host = os.getenv("DB_HOST", "localhost")
port = os.getenv("DB_PORT", "5432")
database = os.getenv("DB_NAME", "customer_behavior")

# Create Engine
engine = create_engine(f"postgresql+psycopg2://{username}:{password}@{host}:{port}/{database}")

# Export cleaned DataFrame
# df.to_sql('customer', engine, if_exists='replace', index=False)
print(f"Database connection configured for '{database}' on {host}:{port}.")